In [ ]:
import time
start_time = time.time()

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix


In [ ]:
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Selected device: {device}")

model_name = "textattack/distilbert-base-uncased-MRPC"
batch_size = 32
max_length = 128
subset_size = 128

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

print(f"Loaded model: {model_name}")
print(f"Batch size: {batch_size}")
print(f"Max length: {max_length}")
print(f"Subset size: {subset_size}")


In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
subset_size = min(subset_size, len(dataset))
dataset = dataset.select(range(subset_size))

print(f"Validation subset examples: {len(dataset)}")

preview_df = dataset.select(range(min(5, len(dataset)))).to_pandas()
print(preview_df[["sentence1", "sentence2", "label"]].to_string(index=False))


In [ ]:
inference_start_time = time.time()

predictions = []
true_labels = []

for start_idx in range(0, len(dataset), batch_size):
    batch = dataset[start_idx:start_idx + batch_size]
    inputs = tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits
        preds = torch.argmax(logits, dim=-1)

    predictions.extend(preds.cpu().tolist())
    true_labels.extend(batch["label"])

inference_seconds = time.time() - inference_start_time
print(f"Completed inference for {len(predictions)} examples in {inference_seconds:.2f} seconds.")


In [ ]:
accuracy = accuracy_score(true_labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels,
    predictions,
    average="binary",
    zero_division=0
)
cm = confusion_matrix(true_labels, predictions)

results_df = pd.DataFrame([
    {
        "model_name": model_name,
        "dataset": "glue/mrpc",
        "split": "validation[:subset]",
        "num_examples": len(dataset),
        "batch_size": batch_size,
        "max_length": max_length,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "device": str(device),
        "inference_seconds": inference_seconds
    }
])

print(results_df.to_string(index=False))
print("Confusion Matrix:")
print(cm)


In [ ]:
examples_df = dataset.to_pandas()[["sentence1", "sentence2", "label"]].copy()
examples_df = examples_df.rename(columns={"label": "true_label"})
examples_df["predicted_label"] = predictions
examples_df["correct"] = examples_df["true_label"] == examples_df["predicted_label"]

print(examples_df.head(10).to_string(index=False))

mismatches_df = examples_df[~examples_df["correct"]].copy()
compact_errors_df = mismatches_df[["sentence1", "sentence2", "true_label", "predicted_label"]].head(10)
print(f"\nMismatches: {len(mismatches_df)}")
if len(compact_errors_df) > 0:
    print(compact_errors_df.to_string(index=False))

elapsed_seconds = time.time() - start_time
print(f"\nTotal runtime (seconds): {elapsed_seconds:.2f}")
